# Outage Dataset: File Exploration

Initial exploration of the EAGLE-I outage dataset structure.

**Source:** `/books/the-grid/data/raw/introduction/Outage_Dataset/`

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../../data/raw/introduction/Outage_Dataset")

## File Inventory

In [2]:
files = sorted(DATA_DIR.glob("*"))

inventory = []
for f in files:
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        inventory.append({
            "filename": f.name,
            "size_mb": round(size_mb, 2),
            "extension": f.suffix
        })

df_inventory = pd.DataFrame(inventory)
print(f"Total files: {len(df_inventory)}")
print(f"Total size: {df_inventory['size_mb'].sum():.1f} MB")
df_inventory

Total files: 51
Total size: 205.5 MB


,filename,size_mb,extension
0,Disclaimer.docx,0.01,.docx
1,eaglei_outages_2014_group.csv,0.02,.csv
2,eaglei_outages_2014_merged.csv,0.87,.csv
3,eaglei_outages_2015_group.csv,0.02,.csv
4,eaglei_outages_2015_merged.csv,7.50,.csv
5,eaglei_outages_2016_group.csv,0.02,.csv
6,eaglei_outages_2016_merged.csv,7.58,.csv
7,eaglei_outages_2017_group.csv,0.02,.csv
8,eaglei_outages_2017_merged.csv,8.17,.csv
9,eaglei_outages_2018_group.csv,0.02,.csv


## File Groupings

The CSV files follow a naming pattern: `eaglei_outages_[variant]_YYYY[_lag].csv`

In [3]:
import re

def classify_file(filename):
    """Extract file type and year from filename."""
    if not filename.endswith(".csv"):
        return {"type": "other", "year": None, "variant": None}
    
    # Pattern: eaglei_outages_[with_events_]YYYY[_group|_merged][_N_hours_lag]
    if "_group" in filename:
        file_type = "group"
    elif "_merged" in filename:
        file_type = "merged"
    elif "_24_hours_lag" in filename:
        file_type = "with_events_24h_lag"
    elif "_8_hours_lag" in filename:
        file_type = "with_events_8h_lag"
    elif "with_events" in filename:
        file_type = "with_events"
    else:
        file_type = "unknown"
    
    year_match = re.search(r"_(20\d{2})", filename)
    year = int(year_match.group(1)) if year_match else None
    
    return {"type": file_type, "year": year}

df_inventory["file_type"] = df_inventory["filename"].apply(lambda x: classify_file(x)["type"])
df_inventory["year"] = df_inventory["filename"].apply(lambda x: classify_file(x)["year"])

df_inventory

,filename,size_mb,extension,file_type,year
0,Disclaimer.docx,0.01,.docx,other,NaN
1,eaglei_outages_2014_group.csv,0.02,.csv,group,2014.0
2,eaglei_outages_2014_merged.csv,0.87,.csv,merged,2014.0
3,eaglei_outages_2015_group.csv,0.02,.csv,group,2015.0
4,eaglei_outages_2015_merged.csv,7.50,.csv,merged,2015.0
5,eaglei_outages_2016_group.csv,0.02,.csv,group,2016.0
6,eaglei_outages_2016_merged.csv,7.58,.csv,merged,2016.0
7,eaglei_outages_2017_group.csv,0.02,.csv,group,2017.0
8,eaglei_outages_2017_merged.csv,8.17,.csv,merged,2017.0
9,eaglei_outages_2018_group.csv,0.02,.csv,group,2018.0


In [4]:
summary = df_inventory.groupby("file_type").agg(
    count=("filename", "count"),
    total_mb=("size_mb", "sum"),
    years=("year", lambda x: f"{x.min()}-{x.max()}" if x.notna().any() else "N/A")
).round(1)

summary

,count,total_mb,years
file_type,,,
group,10,0.2,2014.0-2023.0
merged,10,92.7,2014.0-2023.0
other,1,0.0,N/A
with_events,10,83.3,2014.0-2023.0
with_events_24h_lag,10,16.3,2014.0-2023.0
with_events_8h_lag,10,13.0,2014.0-2023.0


## File Sizes Over Time

Do the datasets grow year-over-year? (More counties reporting, more outages, etc.)

In [5]:
csv_files = df_inventory[df_inventory["year"].notna()].copy()

pivot = csv_files.pivot_table(
    index="year", 
    columns="file_type", 
    values="size_mb", 
    aggfunc="sum"
).round(2)

pivot

file_type,group,merged,with_events,with_events_24h_lag,with_events_8h_lag
year,,,,,
2014.0,0.02,0.87,0.14,0.06,0.05
2015.0,0.02,7.50,2.00,0.66,0.57
2016.0,0.02,7.58,3.46,0.65,0.57
2017.0,0.02,8.17,3.82,0.83,0.73
2018.0,0.02,10.52,6.00,1.77,1.51
2019.0,0.02,10.80,8.66,1.58,1.26
2020.0,0.02,11.40,13.76,2.69,2.18
2021.0,0.02,12.00,18.84,4.46,3.67
2022.0,0.02,12.25,14.75,2.12,1.44


## Schema Preview

Peek at column headers for each file type (using 2023 as sample year).

In [6]:
sample_files = {
    "group": "eaglei_outages_2023_group.csv",
    "merged": "eaglei_outages_2023_merged.csv",
    "with_events": "eaglei_outages_with_events_2023.csv",
    "with_events_8h_lag": "eaglei_outages_with_events_2023_8_hours_lag.csv",
}

for file_type, filename in sample_files.items():
    filepath = DATA_DIR / filename
    if filepath.exists():
        df_sample = pd.read_csv(filepath, nrows=0)
        print(f"\n=== {file_type} ({len(df_sample.columns)} columns) ===")
        print(list(df_sample.columns))


=== group (6 columns) ===
['state', 'year', 'month', 'outage_count', 'max_outage_duration', 'customer_weighted_hours']

=== merged (8 columns) ===
['fips', 'state', 'county', 'start_time', 'duration', 'min_customers', 'max_customers', 'mean_customers']

=== with_events (14 columns) ===
['event_id', 'state_event', 'Datetime Event Began', 'Datetime Restoration', 'Event Type', 'fips', 'state', 'county', 'start_time', 'duration', 'end_time', 'min_customers', 'max_customers', 'mean_customers']

=== with_events_8h_lag (13 columns) ===
['event_id', 'state_event', 'Datetime Event Began', 'Datetime Restoration', 'Event Type', 'fips', 'state', 'county', 'start_time', 'duration', 'min_customers', 'max_customers', 'mean_customers']


In [7]:
print("Row counts (2023 sample files):")
print()

for file_type, filename in sample_files.items():
    filepath = DATA_DIR / filename
    if filepath.exists():
        # Count lines without loading full file
        with open(filepath) as f:
            row_count = sum(1 for _ in f) - 1  # subtract header
        print(f"{file_type}: {row_count:,} rows")

Row counts (2023 sample files):

group: 689 rows
merged: 179,356 rows
with_events: 77,341 rows
with_events_8h_lag: 7,221 rows


## Observations

_Fill in after running:_

- 
- 
- 